# From-Scratch 4-Hidden-Layer MLP (NumPy)

This notebook implements:
- A `Linear` layer with `__call__` (forward) and `backward`
- A `ReLU` activation with forward (`__call__`) and `backward`
- A 4-hidden-layer network with `forward` and `backward`
- A simple training loop with SGD and MSE loss for demonstration.


In [ ]:
import numpy as np
np.random.seed(42)  # for reproducibility


## Linear Layer
Implements `y = x @ W + b` with cached input for backprop.

In [ ]:
class Linear:
    """Linear (fully connected) layer with forward and backward pass
    Parameters:
        in_features: input dimension
        out_features: output dimension
    """
    def __init__(self, in_features, out_features):
        # He initialization (good with ReLU)
        self.weights = np.random.randn(in_features, out_features) * np.sqrt(2.0 / in_features)
        self.bias = np.zeros((1, out_features))
        # caches and grads
        self.input = None
        self.grad_weights = None
        self.grad_bias = None

    def __call__(self, x):
        """Forward pass
        x: (batch_size, in_features)
        returns: (batch_size, out_features)
        """
        self.input = x
        return x @ self.weights + self.bias

    def backward(self, grad_output):
        """
        Backward pass
        grad_output: (batch_size, out_features)
        returns grad_input: (batch_size, in_features)
        """
        # dL/dW = X^T @ dL/dY
        self.grad_weights = self.input.T @ grad_output
        # dL/db = sum over batch
        self.grad_bias = np.sum(grad_output, axis=0, keepdims=True)
        # dL/dX = dL/dY @ W^T
        grad_input = grad_output @ self.weights.T
        return grad_input


## ReLU Activation
Passes positive values, zeros-out negatives.

In [ ]:
class ReLU:
    def __init__(self):
        self.input = None

    def __call__(self, x):
        self.input = x
        return np.maximum(0, x)

    def backward(self, grad_output):
        # derivative is 1 where input > 0 else 0
        return grad_output * (self.input > 0)


## 4-Hidden-Layer Network
Uses: Linear -> ReLU repeated 4 times, then a final Linear.

In [ ]:
class FourHiddenLayerNetwork:
    def __init__(self, input_size, hidden_sizes, output_size):
        assert len(hidden_sizes) == 4, "Must have exactly 4 hidden layers"
        self.fc1 = Linear(input_size, hidden_sizes[0])
        self.relu1 = ReLU()
        self.fc2 = Linear(hidden_sizes[0], hidden_sizes[1])
        self.relu2 = ReLU()
        self.fc3 = Linear(hidden_sizes[1], hidden_sizes[2])
        self.relu3 = ReLU()
        self.fc4 = Linear(hidden_sizes[2], hidden_sizes[3])
        self.relu4 = ReLU()
        self.fc5 = Linear(hidden_sizes[3], output_size)
        self.layers = [self.fc1, self.relu1, self.fc2, self.relu2, self.fc3, self.relu3, self.fc4, self.relu4, self.fc5]

    def forward(self, x):
        out = self.fc1(x); out = self.relu1(out)
        out = self.fc2(out); out = self.relu2(out)
        out = self.fc3(out); out = self.relu3(out)
        out = self.fc4(out); out = self.relu4(out)
        out = self.fc5(out)
        return out

    def backward(self, grad_output):
        grad = self.fc5.backward(grad_output)
        grad = self.relu4.backward(grad)
        grad = self.fc4.backward(grad)
        grad = self.relu3.backward(grad) 
        grad = self.fc3.backward(grad)
        grad = self.relu2.backward(grad) 
        grad = self.fc2.backward(grad)
        grad = self.relu1.backward(grad) 
        grad = self.fc1.backward(grad)
        return grad

    def parameters(self):
        for layer in self.layers:
            if isinstance(layer, Linear):
                yield layer.weights, layer.grad_weights
                yield layer.bias, layer.grad_bias


## Optimizer (SGD) and Utilities

In [ ]:
class SGD:
    def __init__(self, network, lr=1e-2):
        self.network = network
        self.lr = lr

    def step(self):
        for layer in self.network.layers:
            if isinstance(layer, Linear):
                layer.weights -= self.lr * layer.grad_weights
                layer.bias -= self.lr * layer.grad_bias

    def zero_grad(self):
        for layer in self.network.layers:
            if isinstance(layer, Linear):
                layer.grad_weights = np.zeros_like(layer.weights)
                layer.grad_bias = np.zeros_like(layer.bias)

def mse_loss(pred, target):
    # returns (loss_scalar, grad w.r.t pred)
    diff = pred - target
    loss = np.mean(diff ** 2)
    grad = (2.0 / pred.shape[0]) * diff
    return loss, grad


## Train on Synthetic Data (Demo)
We will train the network to fit a random mapping for demonstration purposes.

In [ ]:
# Hyperparameters
input_size = 10
hidden_sizes = [64, 128, 128, 64]
output_size = 5
batch_size = 32
epochs = 50
lr = 0.01

# Create network and optimizer
net = FourHiddenLayerNetwork(input_size, hidden_sizes, output_size)
opt = SGD(net, lr=lr)

# Dummy dataset (inputs and targets)
X = np.random.randn(batch_size, input_size)
Y = np.random.randn(batch_size, output_size)

for epoch in range(1, epochs + 1):
    # Forward
    preds = net.forward(X)
    loss, grad_out = mse_loss(preds, Y)

    # Backward
    net.backward(grad_out)

    # Update
    opt.step()

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | Loss: {loss:.6f}')
